# 12 · Tensor-parallel linear layers (Megatron-style)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/12_tensor_parallel.ipynb) &nbsp; 📖 Concept deck: [Parallelism for inference](https://nilesh-salpe.github.io/visual-ai/inference/08-parallelism/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** column- and row-parallel matmuls over simulated GPUs, and a tensor-parallel MLP that needs exactly one all-reduce.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def gelu(x): return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))

def all_reduce(parts, comm):
    """Sum tensors held by different ranks; log the op and the size of the reduced tensor."""
    comm.append(('all_reduce', parts[0].size)); return sum(parts)

def all_gather(parts, axis, comm):
    comm.append(('all_gather', sum(p.size for p in parts))); return np.concatenate(parts, axis=axis)

r = np.random.default_rng(0)
X = r.normal(size=(8, 32)); W1 = r.normal(0, .2, (32, 128)); W2 = r.normal(0, .2, (128, 32))
print('X [8x32], W1 [32x128], W2 [128x32]; helpers all_reduce(parts, comm), all_gather(parts, axis, comm)')

## Task 1: Column-parallel linear

Split `W` along its **output** columns into `n` shards. Each rank computes `X @ W_i` (no communication).
Return the **list** of per-rank outputs; the caller decides whether to gather them.

In [ ]:
def column_parallel(X, W, n):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_column_parallel(f):
    parts = f(X, W1, 4)
    assert len(parts) == 4 and parts[0].shape == (8, 32), "4 ranks, each [8 x 128/4]"
    assert np.allclose(np.concatenate(parts, 1), X @ W1), "concatenated shards must equal X @ W1"

check("column_parallel", _test_column_parallel, column_parallel)

## Task 2: Row-parallel linear

The input arrives **already split** along its feature dimension: `X_parts[i]` has shape [T, in/n]. Split `W` along its **input** rows to match, compute the partial products `X_i @ W_i`, and **all-reduce** (sum) them with `all_reduce(parts, comm)`.

In [ ]:
def row_parallel(X_parts, W, comm):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_row_parallel(f):
    comm = []; H = gelu(X @ W1)
    y = f(np.split(H, 4, axis=1), W2, comm)
    assert np.allclose(y, H @ W2), "row-parallel result must equal the full matmul"
    assert comm == [('all_reduce', 8 * 32)], f"expected exactly one all-reduce of an 8x32 tensor, got {comm}"

check("row_parallel", _test_row_parallel, row_parallel)

## Task 3: Tensor-parallel MLP with one all-reduce

Compute `gelu(X @ W1) @ W2` across `n` ranks: **column-parallel** W1, apply GELU **locally** on each shard (it's elementwise, so no communication), then **row-parallel** W2. Exactly one all-reduce, no all-gather.

In [ ]:
def tp_mlp(X, W1, W2, n, comm):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_tp_mlp(f):
    comm = []
    assert np.allclose(f(X, W1, W2, 4, comm), gelu(X @ W1) @ W2), "TP MLP must equal the unsplit MLP"
    assert comm == [('all_reduce', 8 * 32)], f"the MLP should need exactly ONE all-reduce, got {comm}"

check("tp_mlp", _test_tp_mlp, tp_mlp)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Column-parallel linear
def column_parallel_solution(X, W, n):
    return [X @ Wi for Wi in np.split(W, n, axis=1)]

check("column_parallel (reference)", _test_column_parallel, column_parallel_solution)

In [ ]:
# @title Solution for Task 2: Row-parallel linear
def row_parallel_solution(X_parts, W, comm):
    W_parts = np.split(W, len(X_parts), axis=0)
    return all_reduce([x @ w for x, w in zip(X_parts, W_parts)], comm)

check("row_parallel (reference)", _test_row_parallel, row_parallel_solution)

In [ ]:
# @title Solution for Task 3: Tensor-parallel MLP with one all-reduce
def tp_mlp_solution(X, W1, W2, n, comm):
    local = [gelu(h) for h in column_parallel_solution(X, W1, n)]   # no communication
    return row_parallel_solution(local, W2, comm)

check("tp_mlp (reference)", _test_tp_mlp, tp_mlp_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
# Per-token communication of TP in decode: one all-reduce of d_model values per (attention or MLP) block per layer.
for name, d, layers in [('8B-shaped', 4096, 32), ('70B-shaped', 8192, 80)]:
    per_tok = 2 * layers * d * 2                    # 2 all-reduces per layer, BF16 bytes
    print(f"{name}: ~{per_tok/1e6:.2f} MB all-reduced per generated token per sequence (before ring factor 2(n-1)/n)")
print("That small, latency-sensitive traffic is why TP stays inside the NVLink domain.")

---
## Takeaways
- Column split → each rank owns some output features. Row split → each rank holds a partial sum that needs an **all-reduce**.
- Column-then-row (Megatron) keeps elementwise ops local, so each MLP or attention block costs **one** all-reduce.
- In decode those all-reduces are small and on the critical path, so TP is kept within NVLink.